In [0]:
df_crm_products = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.crm_products
""")
df_crm_sales = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.crm_sales
""")
df_erp_categories = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.erp_erp_categories
""")
df_erp_customers = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.erp_erp_customers
""")
df_erp_locations = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.erp_erp_locations
""")

In [0]:
from pyspark.sql import functions as F

# Incremental processing: Only process data not yet in silver
target_table = "sqldatawarehouse.silver.crm_products"

try:
    # Check if silver table exists and get max process_timestamp
    max_process_ts = spark.sql(f"""
        SELECT COALESCE(MAX(process_timestamp), CAST('1900-01-01' AS TIMESTAMP)) as max_ts
        FROM {target_table}
    """).collect()[0]['max_ts']
    
    print(f"📊 Incremental mode: Processing data ingested after {max_process_ts}")
    
    # Filter bronze data to only include records newer than max process_timestamp
    initial_count = df_crm_products.count()
    df_crm_products = df_crm_products.filter(F.col("ingestion_time") > F.lit(max_process_ts))
    filtered_count = df_crm_products.count()
    
    print(f"   Initial records from bronze: {initial_count}")
    print(f"   New records to process: {filtered_count}")
    print(f"   Skipped (already processed): {initial_count - filtered_count}")
    
except Exception as e:
    # Table doesn't exist yet - process all data (first run)
    print(f"📊 Full load mode: Silver table does not exist yet, processing all bronze data")
    print(f"   Records to process: {df_crm_products.count()}")

In [0]:
from pyspark.sql import functions as F

result = (
    df_crm_products.groupBy("prd_id")
      .count()
      .filter((F.col("count") > 1) | F.col("prd_id").isNull())
)

result.show()

In [0]:
df_crm_products = df_crm_products.withColumn('cat_id', F.regexp_replace(F.substring('prd_key', 1, 5), '-', '_'))

df_crm_products = df_crm_products.withColumn('prd_key',F.substring('prd_key', 7, F.length('prd_key')))


df_crm_products.select("cat_id", "prd_key").display()

In [0]:
# df_crm_products = df_crm_products.withColumn('prd_cost', F.col('prd_cost').cast('int'))

In [0]:
df_crm_products = df_crm_products.withColumn('prd_cost', F.when(F.col('prd_cost') == '', '0').otherwise(F.col('prd_cost')))

In [0]:
df_crm_products.select("prd_cost").display()

In [0]:
df_crm_products = df_crm_products.withColumn('prd_cost', F.col('prd_cost').cast('int'))

df_crm_products.display()

In [0]:
display(df_crm_products.select('prd_line').distinct())

In [0]:
df_crm_products = df_crm_products.withColumn(
    "prd_start_dt",
    F.when(F.col("prd_start_dt") == "", F.lit(None)).otherwise(F.col("prd_start_dt"))
).withColumn(
    "prd_end_dt",
    F.when(F.col("prd_end_dt") == "", F.lit(None)).otherwise(F.col("prd_end_dt"))
)

In [0]:
df_crm_products = df_crm_products.withColumn(
    "prd_line",
    F.when(F.trim(F.upper(F.col("prd_line"))) == "M", "Mountain")
     .when(F.trim(F.upper(F.col("prd_line"))) == "R", "Road")
     .when(F.trim(F.upper(F.col("prd_line"))) == "S", "Other Sales")
     .when(F.trim(F.upper(F.col("prd_line"))) == "T", "Touring")
     .otherwise("Unknown")
)

In [0]:
df_crm_products = df_crm_products.withColumn('prd_start_dt', F.col('prd_start_dt').cast('date'))
df_crm_products = df_crm_products.withColumn('prd_end_dt', F.col('prd_end_dt').cast('date'))

In [0]:
df_crm_products.display()

In [0]:
df_crm_products.select('prd_key','prd_start_dt', "prd_end_dt").filter(F.col('prd_start_dt') > F.col('prd_end_dt')).display()

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("prd_key").orderBy("prd_start_dt")
#
df_crm_products = df_crm_products.withColumn(
    "corrected_end_dt",
    F.when(
        F.col("prd_start_dt") > F.col("prd_end_dt"),
        F.date_sub(F.lead("prd_start_dt").over(w), 1)
    ).otherwise(F.col("prd_end_dt"))
).drop("prd_end_dt").withColumnRenamed("corrected_end_dt", "prd_end_dt")

In [0]:
# df = df_crm_products.select('prd_key','prd_start_dt', "prd_end_dt").filter(F.col('prd_start_dt') > F.col('prd_end_dt'))

df_crm_products.select('prd_key','prd_start_dt', "prd_end_dt").filter(F.col('prd_key') =='HL-U509-R').display()


In [0]:
df_crm_products.display()

In [0]:
df_crm_products = df_crm_products.withColumn('prd_id', F.col('prd_id').cast('int'))

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp

# Configuration
SILVER_DB = "sqldatawarehouse.silver"
AUDIT_TABLE = f"{SILVER_DB}.transformation_audit"

# Get batch_id from parameter or generate new one (for standalone runs)
try:
    batch_id = dbutils.widgets.get("batch_id")
    print(f"⚙ Using shared batch_id from master pipeline: {batch_id}")
except:
    batch_id = str(uuid.uuid4())
    print(f"⚙ Running standalone - generated new batch_id: {batch_id}")

In [0]:
# Define audit function only if not already defined by master notebook
if 'log_silver_audit' not in dir():
    def log_silver_audit(source_table, target_table, row_count, transformations_applied, status):
        """
        Log silver layer transformation to audit table
        
        Args:
            source_table: Bronze table(s) used as source
            target_table: Silver table written to
            row_count: Number of rows in the result
            transformations_applied: Description of transformations
            status: SUCCESS or FAILED
        """
        audit_df = spark.createDataFrame([(
            source_table,
            target_table,
            row_count,
            batch_id,
            transformations_applied,
            status,
            datetime.now()
        )], [
            "source_table",
            "target_table",
            "row_count",
            "batch_id",
            "transformations_applied",
            "status",
            "transformation_time"
        ])
        
        audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)
    print("✓ Audit function defined")
else:
    print("✓ Using shared audit function from master pipeline")

In [0]:
# Define source and target for products
source_table = "sqldatawarehouse.bronze.crm_products"
target_table = f"{SILVER_DB}.crm_products"

# List of transformations applied to products
transformations = [
    "Incremental processing: Only processed records where bronze ingestion_time > max silver process_timestamp",
    "Extracted cat_id from first 5 characters of prd_key (replacing - with _)",
    "Modified prd_key to substring starting from position 7",
    "Replaced empty prd_cost values with '0'",
    "Cast prd_cost to integer type",
    "Standardized prd_line: M→Mountain, R→Road, S→Other Sales, T→Touring, others→Unknown",
    "Replaced empty prd_start_dt and prd_end_dt with NULL",
    "Cast prd_start_dt and prd_end_dt to date type",
    "Fixed date inconsistencies where prd_start_dt > prd_end_dt using window functions",
    "Cast prd_id to integer type",
    "Added process_timestamp column to track silver layer load time"
]

try:
    # Update batch_id and add process_timestamp for silver layer
    from pyspark.sql import functions as F
    df_crm_products = df_crm_products.withColumns({
        'batch_id': F.lit(batch_id),
        'process_timestamp': F.current_timestamp()
    })
    
    # Get row count
    row_count = df_crm_products.count()
    
    # Write to silver layer (with schema evolution enabled)
    df_crm_products.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(target_table)
    
    print(f"✓ Successfully wrote {row_count} rows to {target_table}")
    
    # Log audit
    log_silver_audit(
        source_table=source_table,
        target_table=target_table,
        row_count=row_count,
        transformations_applied="; ".join(transformations),
        status="SUCCESS"
    )
    
    print(f"✓ Audit logged to {AUDIT_TABLE}")
    print(f"✓ Batch ID: {batch_id}")
    
except Exception as e:
    print(f"✗ Failed to write to {target_table}: {str(e)}")
    
    # Log failure
    log_silver_audit(
        source_table=source_table,
        target_table=target_table,
        row_count=0,
        transformations_applied="; ".join(transformations),
        status=f"FAILED: {str(e)[:200]}"
    )
    
    raise

In [0]:
# Verify the products table
print(f"\nVerifying {target_table}:")
spark.sql(f"DESCRIBE TABLE {target_table}").show(truncate=False)

print(f"\nRow count in {target_table}: {spark.table(target_table).count()}")

# Show recent audit logs for this batch
print(f"\nAudit logs for batch {batch_id}:")
spark.sql(f"""
    SELECT 
        target_table,
        row_count,
        status,
        transformation_time,
        LEFT(transformations_applied, 100) as transformations_preview
    FROM {AUDIT_TABLE}
    WHERE batch_id = '{batch_id}'
    ORDER BY transformation_time
""").show(truncate=False)